# 2.5 — The By-Key Aggregators

**Chapter 2, section 2.6.2** (*When the Output Type Differs from the Input Type*).

**The question this notebook answers:** `reduceByKey` is the operator to reach for — so when
is it the wrong one, and what replaces it?

`reduceByKey` carries a restriction that is easy to miss because summing numbers satisfies it
silently: its combining function takes two values **of the value's own type** and returns that
same type. Values in, same thing out. The moment the result type differs from the value type —
values are strings and the answer is a *set* of strings — the restriction bites, and there are
three ways to respond. One of them is wasteful, and the other two are what
`aggregateByKey` and `combineByKey` exist for.

All three appear below, on the same data, producing the same answer, so that the difference
between them is visible as something other than syntax.

Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
# Chapter 2, section 2.6.2.
import os, time, random, tempfile
from pyspark.sql import SparkSession

SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-2.5")
         .master("local[*]")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext
print("Spark", spark.version, "on", sc.master)

Spark 4.2.0 on local[*]


## 1. The data

Two RDDs zipped into a pair RDD. `zip` pairs the two element by element, and it requires both
sides to have the same number of partitions *and* the same number of elements in each — which
is why it is used here on two RDDs built the same way, and why it is a rare operation in
practice.

In [2]:
valueRDDA = sc.parallelize(["k", "f", "x", "w", "y", "f", "y", "k", "f"])
rddB = sc.parallelize([1, 2, 1, 2, 1, 2, 1, 1, 1])

rdd1 = rddB.zip(valueRDDA)
print("rdd1 :", rdd1.collect())

print("\nThe goal: for each key, the SET of distinct strings observed.")
print("Values are strings.  The answer is a set.  The types differ, and that is the")
print("whole problem this section is about.")

rdd1 : [(1, 'k'), (2, 'f'), (1, 'x'), (2, 'w'), (1, 'y'), (2, 'f'), (1, 'y'), (1, 'k'), (1, 'f')]

The goal: for each key, the SET of distinct strings observed.
Values are strings.  The answer is a set.  The types differ, and that is the
whole problem this section is about.


## 2. `reduceByKey`, forced

It can be made to work: turn each value into a one-element set first, and then the combining
function really does take two sets and return a set. It is sometimes written this way.

In [3]:
forced = (rdd1
          .map(lambda x: (x[0], set(x[1])))            # <-- a brand-new set PER RECORD
          .reduceByKey(lambda a, b: a.union(b)))
print(sorted(forced.collect()))

[(1, {'y', 'f', 'x', 'k'}), (2, {'f', 'w'})]


The answer is right, and the cost is invisible in it. `map(lambda x: (x[0], set(x[1])))`
allocates a fresh one-element set for **every record in the data set**, all of which exist
only to be merged away immediately. At scale that is a large amount of entirely unnecessary
work for the garbage collector.

The cell below makes the allocation count explicit, in plain Python and with no Spark, so that
the counter is not itself distributed across processes. It counts what each strategy would do
to one partition.

In [4]:
class CountingSet(set):
    """A set that records how many times one was constructed."""
    made = 0
    def __init__(self, *a):
        super().__init__(*a)
        type(self).made += 1

_rnd = random.Random(9)
partition = [(i % 4, _rnd.choice("abcdefg")) for i in range(100_000)]

# Strategy A: one set per record, then merge.
CountingSet.made = 0
acc = {}
for k, v in partition:
    s = CountingSet([v])                       # the map(set(...)) step
    acc[k] = acc[k].union(s) if k in acc else s
a_made = CountingSet.made

# Strategy B: one set per key, merged into.
CountingSet.made = 0
acc = {}
for k, v in partition:
    if k not in acc:
        acc[k] = CountingSet()                 # the aggregateByKey zero value
    acc[k].add(v)
b_made = CountingSet.made

print(f"records in the partition          : {len(partition):>8,}")
print(f"sets allocated, one-set-per-record: {a_made:>8,}")
print(f"sets allocated, one-set-per-key   : {b_made:>8,}")
print(f"\nOne strategy allocates once per record, the other once per key: "
      f"{a_made / b_made:,.0f}x here.")
print("The gap widens with the size of the data and not at all with the number of keys,")
print("which is why it is a scaling problem rather than a constant factor.")

records in the partition          :  100,000
sets allocated, one-set-per-record:  100,000
sets allocated, one-set-per-key   :        4

One strategy allocates once per record, the other once per key: 25,000x here.
The gap widens with the size of the data and not at all with the number of keys,
which is why it is a scaling problem rather than a constant factor.


## 3. `aggregateByKey`

`aggregateByKey` exists for precisely this case. It takes three things:

* a **zero value** of the *result* type — here, an empty set;
* **`seqFunc`**, which merges one raw value into an accumulator — the map-side combine;
* **`combFunc`**, which merges two accumulators — the reduce-side merge.

The accumulator is created once per key per partition, not once per record.

In [5]:
def merge_value(acc, v):        # accumulator + one raw value -> accumulator
    acc.add(v)
    return acc

def merge_combiners(a, b):      # accumulator + accumulator   -> accumulator
    return a.union(b)

by_set = rdd1.aggregateByKey(set(), merge_value, merge_combiners)
print(sorted(by_set.collect()))

[(1, {'y', 'f', 'x', 'k'}), (2, {'f', 'w'})]


In [6]:
# The chapter's own example, at section 2.6.2, verbatim -- so the notebook and the notes
# can be checked against each other.
chapter_rdd = sc.parallelize([(1, 'k'), (2, 'f'), (1, 'x'), (2, 'w'), (1, 'y')])
print(sorted(chapter_rdd.aggregateByKey(set(), merge_value, merge_combiners).collect()))

[(1, {'y', 'k', 'x'}), (2, {'f', 'w'})]


In [7]:
# The same operator with a list accumulator instead of a set: order is kept and duplicates
# survive.  The zero value determines what "aggregate" means here.
def add_to_list(acc, v):
    acc.append(v)
    return acc

def extend_lists(a, b):
    a.extend(b)
    return a

print(sorted(rdd1.aggregateByKey(list(), add_to_list, extend_lists).collect()))

[(1, ['k', 'x', 'y', 'y', 'k', 'f']), (2, ['f', 'w', 'f'])]


> **A trap worth naming.** `seqFunc` and `combFunc` above **mutate** the accumulator and then
> return it. That is the idiomatic form and it is what makes the operator cheap, but it is only
> safe because the accumulator was created from the zero value for this key and this partition.
> Never mutate something that came from outside — and note that Python's default-argument and
> closure rules make it easy to share one list by accident.

## 4. `combineByKey`

`combineByKey` is the same idea with one additional function. Where `aggregateByKey` starts
from a **zero value** you supply, `combineByKey` starts by calling **`createCombiner`** on the
*first value seen for each key*. That matters when there is no natural empty accumulator to
begin from — when the accumulator must be built out of a value rather than added to.

Three functions:

* `createCombiner: V -> C` — turn the first value into an accumulator;
* `mergeValue: (C, V) -> C` — fold a further value in;
* `mergeCombiners: (C, C) -> C` — merge two accumulators.

In [8]:
def to_set(v):                  # createCombiner: the FIRST value becomes the accumulator
    return {v}

by_combine = rdd1.combineByKey(to_set, merge_value, merge_combiners)
print(sorted(by_combine.collect()))

[(1, {'y', 'f', 'x', 'k'}), (2, {'f', 'w'})]


In [9]:
# ...and the list version, which is where createCombiner earns its place: [v] is built out
# of the value, not from an empty accumulator.
print(sorted(rdd1.combineByKey(lambda v: [v], add_to_list, extend_lists).collect()))

[(1, ['k', 'x', 'y', 'y', 'k', 'f']), (2, ['f', 'w', 'f'])]


In [10]:
# All three strategies must agree.  An assertion that runs beats a sentence saying so.
expected = sorted((k, sorted(v)) for k, v in forced.collect())
for name, result in (("aggregateByKey", by_set), ("combineByKey", by_combine)):
    got = sorted((k, sorted(v)) for k, v in result.collect())
    assert got == expected, (name, got, expected)
    print(f"{name:16s} agrees with the forced reduceByKey")

aggregateByKey   agrees with the forced reduceByKey


combineByKey     agrees with the forced reduceByKey


## 5. The cost, at a size where it shows

Nine records cannot demonstrate an allocation problem. The same three strategies on two
million.

In [11]:
N = 2_000_000
KEYS = 40

def gen(idx, it):
    rnd = random.Random(500 + idx)
    for _ in it:
        yield (rnd.randrange(KEYS), rnd.choice("abcdefghijklmnopqrstuvwxyz"))

big = sc.range(0, N, numSlices=8).mapPartitionsWithIndex(gen).cache()
print(f"{big.count():,} records, {KEYS} keys")

def timed(fn):
    t = time.perf_counter()
    out = fn()
    return out, time.perf_counter() - t

r_forced, t_forced = timed(lambda: sorted(
    big.map(lambda x: (x[0], set(x[1]))).reduceByKey(lambda a, b: a.union(b)).collect()))
r_agg, t_agg = timed(lambda: sorted(
    big.aggregateByKey(set(), merge_value, merge_combiners).collect()))
r_comb, t_comb = timed(lambda: sorted(
    big.combineByKey(to_set, merge_value, merge_combiners).collect()))

assert ([(k, sorted(v)) for k, v in r_forced]
        == [(k, sorted(v)) for k, v in r_agg]
        == [(k, sorted(v)) for k, v in r_comb])
print("all three agree on", len(r_agg), "keys\n")

print(f"{'strategy':34s}{'seconds':>9s}")
print(f"{'map(set) + reduceByKey(union)':34s}{t_forced:>9.2f}")
print(f"{'aggregateByKey':34s}{t_agg:>9.2f}")
print(f"{'combineByKey':34s}{t_comb:>9.2f}")
print(f"\nforced / aggregateByKey = {t_forced / t_agg:.1f}x")
print("\nThese are laptop timings and machine-dependent.  The structural fact behind them")
print("is not: the forced form allocates one temporary set per record, so its cost scales")
print("with the number of RECORDS, while the other two scale with the number of KEYS.")

2,000,000 records, 40 keys


all three agree on 40 keys

strategy                            seconds
map(set) + reduceByKey(union)          0.18
aggregateByKey                         0.08
combineByKey                           0.09

forced / aggregateByKey = 2.2x

These are laptop timings and machine-dependent.  The structural fact behind them
is not: the forced form allocates one temporary set per record, so its cost scales
with the number of RECORDS, while the other two scale with the number of KEYS.


## Conclusion

| | starts from | use it when |
|---|---|---|
| `reduceByKey(f)` | nothing — `f: (V, V) -> V` | the result type *is* the value type. Summing, max, min. |
| `aggregateByKey(zero, seqFunc, combFunc)` | a **zero value** you supply | the result type differs and there is a natural empty accumulator: a set, a list, a `(sum, count)` pair |
| `combineByKey(create, mergeValue, mergeCombiners)` | the **first value** for each key | the result type differs and the accumulator must be built *out of* a value |

All three do a **map-side combine** — they reduce within each partition before shuffling — and
that is what separates them from `groupByKey`, which does not reduce at all. Notebook 2.4
measures what that difference costs.

Two cautions to carry forward.

**The zero value must be an identity element.** `aggregateByKey`'s zero is applied once per
partition, so if `combFunc(zero, x) != x` the answer depends on how the data happened to be
partitioned — silently, and differently on a laptop than on a cluster. Notebook 2.6 is entirely
about that failure.

**Choosing `aggregateByKey` does not by itself buy safety.** Chapter 5 makes the point sharply:
these operators are no better than `groupByKey` if the accumulator grows with every record
added. The test to apply before writing any aggregation is whether the *combined accumulator is
smaller than the sum of its parts*. A set of distinct strings passes, because merging collapses
duplicates. A list of every value does not — `aggregateByKey(list(), append, extend)`, shown
above for contrast, is `groupByKey` with extra steps.

**Next.** Notebook 2.6 takes the zero value apart, and shows the same data and the same
functions returning three different answers.